In [ ]:
import csv

import altair as alt
from keyness import log_likelihood
import nltk
from nltk.corpus import stopwords
from nltk.text import Text
from nltk.util import bigrams
import pandas as pd


# nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
alt.data_transformers.disable_max_rows()


DATA_ROOT = '../data'
LARGE_CORPUS = f'{DATA_ROOT}/large_corpus.csv'
LARGE_CORPUS_SCORED = f'{DATA_ROOT}/large_corpus_scored.csv'


In [ ]:
scored_df = pd.read_csv(LARGE_CORPUS_SCORED, quoting=csv.QUOTE_ALL, keep_default_na=False)
large_df = pd.read_csv(LARGE_CORPUS, quoting=csv.QUOTE_ALL, keep_default_na=False)

In [ ]:
category_order = ['negative', 'neutral', 'positive']

# Map ratings to sentiment categories
def rating_to_sentiment(rating):
    if rating <= 1:
        return 'negative'
    elif rating >= 5:
        return 'positive'
    else:
        return 'neutral'

large_df['sentiment'] = large_df['ratings'].apply(rating_to_sentiment)

# Left chart - ratings distribution
ratings_chart = alt.Chart(large_df).mark_bar().encode(
    x=alt.X('sentiment:N', sort=category_order, title='Sentiment (from ratings)'),
    y=alt.Y('count()', title='Count')
).properties(
    width=400,
    height=400,
    title='Ratings Distribution'
)

# Right chart - scored distribution
scored_chart = alt.Chart(scored_df).mark_bar().encode(
    x=alt.X('score:N', sort=category_order, title='Sentiment'),
    y=alt.Y('count()', title='Count')
).properties(
    width=400,
    height=400,
    title='Model Predictions'
)

(ratings_chart | scored_chart).interactive()

Generate bigrams for negative, neutral and positive reviews

In [ ]:
def review_to_bigrams(review: str) -> list[tuple[str, str]]:
    tokenised_review = [token for token in nltk.tokenize.word_tokenize(review)]
    tokenised_review = filter(str.isalpha, tokenised_review)
    tokenised_review = map(str.lower, tokenised_review)   
    tokenised_review = list(filter(lambda e: e.lower() not in stop_words, tokenised_review))
    bigrams_review = list(bigrams(tokenised_review))
    return tokenised_review, bigrams_review


def clean_df(df: pd.DataFrame):
    df = df.copy()
    df.dropna()
    df = df[df['reviews'].apply(lambda e: isinstance(e, str))]
    return df  


def likelihood_filename(sentiment: str) -> str:
    return f"{DATA_ROOT}/{sentiment}_loglikelihood_alternative.tsv"
    

def create_log_likelihood_files(reviews_with_bigrams_df: pd.DataFrame, sentiment: str):
    reference_corpus = reviews_with_bigrams_df[reviews_with_bigrams_df['score']=='neutral']['bigrams']
    corpus = reviews_with_bigrams_df[reviews_with_bigrams_df['score'] == sentiment]['bigrams']
    filename = likelihood_filename(sentiment)
    log_likelihood(corpus=corpus, reference_corpus=reference_corpus,save_as=filename)


reviews_with_bigrams_df = clean_df(scored_df)
reviews_with_bigrams_df[['tokens', 'bigrams']] = reviews_with_bigrams_df.apply( 
    lambda row: review_to_bigrams(row['reviews']), 
    result_type='expand',
    axis=1
)
create_log_likelihood_files(reviews_with_bigrams_df,'negative')
create_log_likelihood_files(reviews_with_bigrams_df,'positive')



In [ ]:
def get_top_150_bigrams(sentiment: str):
    filename = likelihood_filename(sentiment)
    df = pd.read_csv(filename, quoting=csv.QUOTE_ALL, sep='\t', keep_default_na=False)
    df = df.rename(columns={
        'Word': 'bigram', 
        'LL': 'log likelihood', 
        'CC': 'corpus frequency', 
        'RCC': 'reference frequency'
    })
    df = df.sort_values('log likelihood', ascending=False)
    df = df.head(150)
    df['sentiment'] = sentiment
    
    return df



negative_sentiment_df = get_top_150_bigrams('negative')
positive_sentiment_df = get_top_150_bigrams('positive')

In [ ]:
negative_sentiment_df

In [ ]:
positive_sentiment_df

In [ ]:
def create_scatter_chart(df: pd.DataFrame, title: str):
    return alt.Chart(df).mark_circle().encode(
        x=alt.X('reference frequency:Q', title='Reference Frequency'),
        y=alt.Y('log likelihood:Q', title='Log Likelihood'),
        tooltip=['bigram', 'log likelihood', 'reference frequency']
    ).properties(
        width=300,
        height=300,
        title=title
    )

neg_chart = create_scatter_chart(negative_sentiment_df, 'Negative')
pos_chart = create_scatter_chart(positive_sentiment_df, 'Positive')

(neg_chart | pos_chart).interactive()

In [ ]:
negative_tokens = reviews_with_bigrams_df[reviews_with_bigrams_df['score']=='negative']['tokens']
negative_token_list = [token for tokens in negative_tokens for token in tokens]
negative_text = Text(negative_token_list)

In [ ]:
negative_text.concordance(['great', 'game'])

In [ ]:
negative_text.concordance(['works', 'well'])